# MS-COCOAI — Colab (free tier)

Run the cells top to bottom. One-time setup: internet ON, GPU (T4), Drive mounted.

**Dataset:** keep the **17 parquet shards** of the MS-COCOAI dataset in your Google Drive
(e.g. `MyDrive/mscocoai/data/*.parquet`). Large parquets are fine over Drive;
96,000 small files are not.

**Every run checkpoints automatically** and stops cleanly with `--max-hours`
before Colab kills the runtime. If it disconnects, just re-run the train cell —
`--resume auto` continues from the last intact epoch.

In [ ]:
# 1. Install dependencies
!pip install -q -r requirements.txt


In [ ]:
# 2. Clone the code (skip if already in /content/mscocoai)
# !git clone https://github.com/AnnusShahzad/DeepFakeImageDetection.git /content/mscocoai
# %cd /content/mscocoai


In [ ]:
# 3. Mount Drive
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/mscocoai'   # your Drive folder with data/*.parquet inside


In [ ]:
# 4. Preflight: GPU, disk, internet
!python -m mscocoai.doctor --out-dir {DRIVE}/outputs


## Train

Recommended first run: the frequency-aware multi-task configuration.

- `--data-layout parquet` reads the shards directly from Drive
- `--drive-backend copy` copies each epoch's fp16 weights into Drive (no keys)
- `--max-hours 8` stops cleanly before a runtime reset
- `--resume auto` continues after any interruption


In [ ]:
# 5. Train (first run downloads nothing — Drive already holds the parquets)
!python -m mscocoai.train \
    --task multitask --backbone convnext_tiny --freq-branch dct \
    --data-dir {DRIVE} --data-layout parquet \
    --num-workers 8 --batch-size 32 --accum-steps 2 \
    --max-hours 8 --ema-decay 0.999 --label-smoothing 0.05 \
    --drive-backend copy --drive-dir {DRIVE}/outputs \
    --out-dir {DRIVE}/outputs/mt_freq \
    --resume auto


## Resume after a disconnect

Re-run cell 1 (install), then re-run cell 5. `--resume auto` restores the last
epoch automatically; the Drive backup guarantees it exists.


## Evaluate (final, once)

Run this **after** training is complete; the test split is read exactly once for the final report.


In [ ]:
# 6. Final test evaluation + report artifacts
!python -m mscocoai.evaluate \
    --checkpoint {DRIVE}/outputs/mt_freq/best.pt \
    --split test --report-dir {DRIVE}/reports/mt_freq_test
